In [1]:
from torch import nn
import torch
from math import sqrt
from typing import Union
from torch.nn.init import calculate_gain   

warmup_function = {
    "smooth": lambda c, w: min(1.0, c / w),
    "non_smooth": lambda c, w: min(1.0, c // w),
}

def call_reinit(m, i, o):
    m.reinit()


def log_features(m, i, o):
    with torch.no_grad():
        if m.decay_rate == 0:
            m.features = i[0]
        else:
            if m.features is None:
                m.features = (1 - m.decay_rate) * i[0]
            else:
                m.features = m.features * m.decay_rate + (1 - m.decay_rate) * i[0]


def get_layer_bound(layer, init, gain):
    if isinstance(layer, nn.Conv2d):
        return sqrt(1 / (layer.in_channels * layer.kernel_size[0] * layer.kernel_size[1]))
    elif isinstance(layer, nn.Linear):
        if init == 'default':
            bound = sqrt(1 / layer.in_features)
        elif init == 'xavier':
            bound = gain * sqrt(6 / (layer.in_features + layer.out_features))
        elif init == 'lecun':
            bound = sqrt(3 / layer.in_features)
        elif init == 'orthogonal':
            # Simulated uniform bound for orthogonal: entries ~ gain/sqrt(fan_in)
            bound = gain / sqrt(layer.in_features)
        else:
            bound = gain * sqrt(3 / layer.in_features)
        return bound


class BatchRenorm(nn.Module):
    """
    BatchRenorm - (arxiv.org/abs/1702.03275)

    Args:
        num_features: number of features in input tensor
        momentum: momentum for running statistics
        warmup: number of batches to warmup the batch renorm
        max_r: maximum value for r
        max_d: maximum value for d
        smoothing: smoothing factor for transition from BN to BR
    """

    def __init__(
        self,
        num_features,
        momentum=0.01,
        warmup=100000,
        max_r=3.0,
        max_d=5.0,
        warmup_type="smooth",
    ):
        super(BatchRenorm, self).__init__()
        self.momentum = momentum
        self.warmup = warmup
        self.max_r = max_r
        self.max_d = max_d
        self.smoothing = warmup_function[warmup_type]
        self.batch_size = 0
        self.num_features = num_features
        self.register_buffer("step", torch.zeros(1))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.eps = 1e-5

    def forward(self, x: torch.Tensor):
        if not x.dim() >= 2:  # first dim is batch size, second dim is num_features
            raise ValueError("expected 2D input (got {}D input)".format(x.dim()))

        # prepare dimensions for scaling and shifting parameters
        # suppose we have input of shape (batch_size, num_features, height, width) (e.g. (32, 3, 64, 64))
        view_shape = [1, x.shape[1]] + [1] * (x.dim() - 2)  # [1, 3, 1, 1]

        dims = [i for i in range(x.dim()) if i != 1]  # [0, 2, 3]

        running_std = (self.running_var + self.eps).sqrt()

        if self.training:
            mean = x.mean(dims)
            var = x.var(dims, unbiased=False)
            std = (var + self.eps).sqrt()

            r = torch.clamp(std / running_std, 1 / self.max_r, self.max_r)
            d = torch.clamp(
                (mean - self.running_mean) / running_std, -self.max_d, self.max_d
            )

            if self.step < self.warmup:
                # BatchNorm
                smoothing_factor = self.smoothing(self.step.item(), self.warmup)
                r = 1.0 + (r - 1.0) * smoothing_factor
                d = d * smoothing_factor

            # update running statistics
            x = (x - mean.view(view_shape)) / std.view(view_shape) * r.view(
                view_shape
            ) + d.view(view_shape)

            raw_var = var.detach() * x.shape[0] / (x.shape[0] - 1)
            self.running_mean += self.momentum * (mean.detach() - self.running_mean)
            self.running_var += self.momentum * (raw_var - self.running_var)

            self.step += 1

        else:
            # inference time
            x = (x - self.running_mean.view(view_shape)) / running_std.view(view_shape)

        return x * self.weight.view(view_shape) + self.bias.view(view_shape)

class CBPConv1d(nn.Module):
    """
    CBPConv1d Module
    This module implements a 1D convolutional block with Continual Bacculate Plasticity to enhance 
    plasticity in continual learning scenarios. It dynamically manages feature utility and performs
    selective reinitialization to maintain network plasticity over time.
    
    Initialization Parameters:
        in_layer (nn.Conv1d): Input 1D convolutional layer used to extract features.
        out_layer (Union[nn.Conv1d, nn.Linear]): Output layer that processes features, can be either Conv1d or Linear.
        ln_layer (nn.LayerNorm, optional): Layer normalization applied to the output features.
        bn_layer (nn.BatchNorm1d, optional): Batch normalization applied to the output features.
        num_last_filter_outputs (int, optional): Specifies the number of output channels for the final filter outputs.
        replacement_rate (float, optional): Defines the rate at which underperforming features are reinitialized.
        maturity_threshold (int, optional): The minimum age a feature must reach before it becomes eligible for reinitialization.
        init (str, optional): Initialization method for weights (e.g., 'kaiming').
        act_type (str, optional): Activation type utilized to determine the gain for weight initialization.
        util_type (str, optional): Strategy employed to compute feature utility.
        decay_rate (float, optional): Decay factor applied to the utility metric during updates.
    
    Usage:
        Define the necessary parameters during initialization, and implement the forward method
        to specify the data flow through the 1D convolutional transformation.
    
    ----------------------------------------------------------------------------
    Based on: Loss of Plasticity in Deep Continual Learning
    ----------------------------------------------------------------------------
    """
    def __init__(
            self,
            in_layer: nn.Conv1d,
            out_layer: Union[nn.Conv1d, nn.Linear],
            ln_layer: nn.LayerNorm = None,
            bn_layer: BatchRenorm = None,
            num_last_filter_outputs=1,
            replacement_rate=1e-5,
            maturity_threshold=1000,
            init='kaiming',
            act_type='relu6',
            util_type='contribution',
            decay_rate=0,
    ):
        super().__init__()
        if type(in_layer) is not nn.Conv1d:
            raise Warning("Make sure in_layer is a 1D convolutional layer")
        if type(out_layer) not in [nn.Linear, nn.Conv1d]:
            raise Warning("Make sure out_layer is a convolutional or linear layer")

        """
        Define the hyper-parameters of the algorithm
        """
        self.replacement_rate = replacement_rate
        self.maturity_threshold = maturity_threshold
        self.util_type = util_type
        self.decay_rate = decay_rate
        self.features = None
        self.num_last_filter_outputs = num_last_filter_outputs

        """
        Register hooks
        """
        if self.replacement_rate > 0:
            self.register_full_backward_hook(call_reinit)
            self.register_forward_hook(log_features)

        self.in_layer = in_layer
        self.out_layer = out_layer
        self.ln_layer = ln_layer
        self.bn_layer = bn_layer
        """
        Utility of all features/neurons
        """
        self.util = nn.Parameter(torch.zeros(self.in_layer.out_channels), requires_grad=False)
        self.ages = nn.Parameter(torch.zeros(self.in_layer.out_channels), requires_grad=False)
        self.accumulated_num_features_to_replace = nn.Parameter(torch.zeros(1), requires_grad=False)
        """
        Calculate uniform distribution's bound for random feature initialization
        """
        self.bound = get_layer_bound(layer=self.in_layer, init=init, gain=calculate_gain(nonlinearity=act_type))

    def forward(self, _input):
        return _input

    def get_features_to_reinit(self):
        """
        Returns: Features to replace
        """
        features_to_replace_input_indices = torch.empty(0, dtype=torch.long, device=self.util.device)
        features_to_replace_output_indices = torch.empty(0, dtype=torch.long, device=self.util.device)
        self.ages += 1
        """
        Calculate number of features to replace
        """
        eligible_feature_indices = torch.where(self.ages > self.maturity_threshold)[0]
        if eligible_feature_indices.shape[0] == 0:  return features_to_replace_input_indices, features_to_replace_output_indices

        num_new_features_to_replace = self.replacement_rate*eligible_feature_indices.shape[0]
        self.accumulated_num_features_to_replace += num_new_features_to_replace
        if self.accumulated_num_features_to_replace < 1:    return features_to_replace_input_indices, features_to_replace_output_indices

        num_new_features_to_replace = int(self.accumulated_num_features_to_replace)
        self.accumulated_num_features_to_replace -= num_new_features_to_replace
        """
        Calculate feature utility
        """
        if isinstance(self.out_layer, torch.nn.Linear):
            output_weight_mag = self.out_layer.weight.data.abs().mean(dim=0).view(-1, self.num_last_filter_outputs)
            self.util.data = (output_weight_mag * self.features.abs().mean(dim=0).view(-1, self.num_last_filter_outputs)).mean(dim=1)
        elif isinstance(self.out_layer, torch.nn.Conv1d):
            output_weight_mag = self.out_layer.weight.data.abs().mean(dim=(0, 2))
            self.util.data = output_weight_mag * self.features.abs().mean(dim=(0, 2))
        """
        Find features with smallest utility
        """
        new_features_to_replace = torch.topk(-self.util[eligible_feature_indices], num_new_features_to_replace)[1]
        new_features_to_replace = eligible_feature_indices[new_features_to_replace]
        features_to_replace_input_indices, features_to_replace_output_indices = new_features_to_replace, new_features_to_replace

        if isinstance(self.in_layer, torch.nn.Conv1d) and isinstance(self.out_layer, torch.nn.Linear):
            features_to_replace_output_indices = (
                    (new_features_to_replace * self.num_last_filter_outputs).repeat_interleave(self.num_last_filter_outputs) +
                    torch.tensor([i for i in range(self.num_last_filter_outputs)]).repeat(new_features_to_replace.size()[0]))
        return features_to_replace_input_indices, features_to_replace_output_indices

    def reinit_features(self, features_to_replace_input_indices, features_to_replace_output_indices):
        """
        Reset input and output weights for low utility features
        """
        with torch.no_grad():
            num_features_to_replace = features_to_replace_input_indices.shape[0]
            if num_features_to_replace == 0: return
            self.in_layer.weight.data[features_to_replace_input_indices, :] *= 0.0
            # noinspection PyArgumentList
            self.in_layer.weight.data[features_to_replace_input_indices, :] += \
                torch.empty([num_features_to_replace] + list(self.in_layer.weight.shape[1:]), device=self.util.device).uniform_(-self.bound, self.bound)
            self.in_layer.bias.data[features_to_replace_input_indices] *= 0

            self.out_layer.weight.data[:, features_to_replace_output_indices] = 0
            self.ages[features_to_replace_input_indices] = 0

            """
            Reset the corresponding batchnorm/layernorm layers
            """
            if self.bn_layer is not None:
                self.bn_layer.bias.data[features_to_replace_input_indices] = 0.0
                self.bn_layer.weight.data[features_to_replace_input_indices] = 1.0
                self.bn_layer.running_mean.data[features_to_replace_input_indices] = 0.0
                self.bn_layer.running_var.data[features_to_replace_input_indices] = 1.0
            if self.ln_layer is not None:
                self.ln_layer.bias.data[features_to_replace_input_indices] = 0.0
                self.ln_layer.weight.data[features_to_replace_input_indices] = 1.0

    def reinit(self):
        """
        Perform selective reinitialization
        """
        features_to_replace_input_indices, features_to_replace_output_indices = self.get_features_to_reinit()
        self.reinit_features(features_to_replace_input_indices, features_to_replace_output_indices)

class Encoder(nn.Module):
    """
    this is the encoding module
    """

    def __init__(
        self,
        input_dim,
        num_layers=2,
        hidden_size=512,
        history_length=1,
        concat_action=False,
        dropout=0.0,
    ):
        super().__init__()
        self.hidden_size = self.feature_dim = hidden_size

    def get_feature_dim(self):
        return self.feature_dim
    
    def forward(self, states, actions=None):
        return None


In [2]:
class TCNEncoder(Encoder):
    def __init__(
        self,
        input_dim,
        num_layers=2,
        hidden_size=512,
        history_length=11,
        kernel_size=11,
        use_continual_backprop=False,
        batch_norm=False,
        #dropout=0.2,
    ):
        super().__init__(
            input_dim,
            num_layers,
            hidden_size,
            history_length=history_length,
            #dropout=dropout,
        )
        self.feature_dim = input_dim[0]
        self.hidden_size = hidden_size
        layers = []
        dilation = 1
        
        for i in range(num_layers):
            in_ch = input_dim[0] if i == 0 else hidden_size
            padding = (dilation * (kernel_size - 1) + 1) // 2
            
            conv_layer = nn.Conv1d(
                        in_ch,
                        hidden_size,
                        kernel_size,
                        dilation=dilation,
                        padding=padding,
                    )
            
            bn_layer = None
            if batch_norm:
                bn_layer = BatchRenorm(hidden_size)
                layers.append(bn_layer)
                
            layers.append(conv_layer)
            layers.append(nn.ReLU())
            
            next_conv = None
            if i < num_layers - 1:
                next_conv = nn.Conv1d(
                    hidden_size,
                    hidden_size,
                    kernel_size,
                    dilation=dilation*2,
                    padding=(dilation*2 * (kernel_size - 1) + 1) // 2,
                )
            elif i == num_layers - 1:
                next_conv = nn.Conv1d(hidden_size, input_dim[0], padding=0, kernel_size=1) #bottleneck layer
            
            if use_continual_backprop and next_conv is not None:
                cbp_layer = CBPConv1d(
                    in_layer=conv_layer,
                    out_layer=next_conv,
                    bn_layer=bn_layer,
                    act_type='relu',
                )
                layers.append(cbp_layer)
            
            dilation *= 2 # to expand the receptive field
            
        bottleneck_layer = nn.Conv1d(hidden_size, input_dim[0], padding=0, kernel_size=1)
        layers.append(bottleneck_layer) # Bottleneck layer
        
        self.net = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
    def forward(self, x):
        x1 = self.net(x)
        x = x + x1
        x = x.permute(0, 2, 1) 
        x = self.global_pool(x)
        return x.squeeze(-1)

In [3]:
lay = TCNEncoder(
    input_dim=(3, 64),
    num_layers=2,
    hidden_size=512,
    history_length=11,
    kernel_size=11,
    use_continual_backprop=True,
    batch_norm=False,
)
x = torch.randn(32, 3, 64)
lay(x)
# Create a random target tensor the same shape as the output
output = lay(x)
print(f"Output shape: {output.shape}")

# Create a dummy loss function (e.g., MSE with random target)
target = torch.randn_like(output)
loss_fn = torch.nn.MSELoss()
loss = loss_fn(output, target)

# Perform backward pass
loss.backward()

# Verify that gradients are computed
print(f"Loss value: {loss.item()}")
print(f"Has gradient for input layer weights: {lay.net[0].weight.grad is not None}")
print(f"CBP layer 1 feature shape: {lay.net[2].features.shape if lay.net[2].features is not None else None}")
print(f"CBP layer 2 feature shape: {lay.net[5].features.shape if lay.net[5].features is not None else None}")

Output shape: torch.Size([32, 64])
Loss value: 1.3384864330291748
Has gradient for input layer weights: True
CBP layer 1 feature shape: torch.Size([32, 512, 64])
CBP layer 2 feature shape: torch.Size([32, 512, 64])


In [4]:
lay

TCNEncoder(
  (net): Sequential(
    (0): Conv1d(3, 512, kernel_size=(11,), stride=(1,), padding=(5,))
    (1): ReLU()
    (2): CBPConv1d(
      (in_layer): Conv1d(3, 512, kernel_size=(11,), stride=(1,), padding=(5,))
      (out_layer): Conv1d(512, 512, kernel_size=(11,), stride=(1,), padding=(10,), dilation=(2,))
    )
    (3): Conv1d(512, 512, kernel_size=(11,), stride=(1,), padding=(10,), dilation=(2,))
    (4): ReLU()
    (5): CBPConv1d(
      (in_layer): Conv1d(512, 512, kernel_size=(11,), stride=(1,), padding=(10,), dilation=(2,))
      (out_layer): Conv1d(512, 3, kernel_size=(1,), stride=(1,))
    )
    (6): Conv1d(512, 3, kernel_size=(1,), stride=(1,))
  )
  (global_pool): AdaptiveAvgPool1d(output_size=1)
)